# Day 1 — Run Your First AI Model

**Goal:** load a small pretrained language model and make it answer prompts.

**Before you start (Colab):** `Runtime → Change runtime type → T4 GPU → Save`. A CPU runtime also works, just slower.

Run each cell in order with **Shift + Enter**.

## Step 1 — Install / upgrade the libraries
Colab already has PyTorch. This makes sure `transformers` and `accelerate` are recent enough.

In [ ]:
!pip install -q -U "transformers>=4.56" accelerate

## Step 2 — Check your hardware
`cuda` = NVIDIA GPU, `xpu` = Intel GPU, `mps` = Apple GPU, `cpu` = no GPU.

In [ ]:
import torch

def pick_device():
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch, "xpu") and torch.xpu.is_available():
        return "xpu"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

device = pick_device()
print("PyTorch:", torch.__version__)
print("Device :", device)
if device == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory free/total: {free/1e9:.1f} / {total/1e9:.1f} GB")

## Step 3 — Load the tokenizer and the model
First run downloads ~1 GB from Hugging Face. No account or token is needed for this model.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype="auto").to(device)
model.eval()

print("Parameters:", round(sum(p.numel() for p in model.parameters()) / 1e6), "million")
print("Weights dtype:", model.dtype)

## Step 4 — Peek inside: text → tokens → text
The model never sees letters, only numbers (token IDs).

In [ ]:
text = "Fine-tuning makes models useful."
ids = tokenizer(text)["input_ids"]
print("Token IDs:", ids)
print("Pieces   :", tokenizer.convert_ids_to_tokens(ids))
print("Back     :", tokenizer.decode(ids))

## Step 5 — A reusable `ask()` function
We wrap the whole generate process in one function so experiments are one line each.

In [ ]:
import time

def ask(prompt, max_new_tokens=120, do_sample=True, temperature=0.7, top_p=0.9, seed=42):
    torch.manual_seed(seed)
    messages = [
        {"role": "system", "content": "You are a helpful, concise assistant."},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(device)

    settings = {"max_new_tokens": max_new_tokens, "do_sample": do_sample}
    if do_sample:
        settings.update(temperature=temperature, top_p=top_p)

    start = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, **settings)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    print(answer)
    print(f"\n[{len(new_tokens)} tokens, {time.time()-start:.1f}s]")
    return answer

_ = ask("Explain what a pretrained language model is in two sentences.")

## Step 6 — Try your own prompts

In [ ]:
_ = ask("Give me three tips for reducing the cost of an injection-moulded plastic part.")

In [ ]:
_ = ask("Write a one-line motto for a small bakery.")

## Experiment A — `max_new_tokens` (answer length cap)
Watch the answer get cut off mid-sentence when the cap is small.

In [ ]:
for n in [20, 60, 200]:
    print(f"\n===== max_new_tokens = {n} =====")
    _ = ask("Describe how a car paint shop works.", max_new_tokens=n)

## Experiment B — `temperature` (randomness)
Low = safe and repetitive. High = varied, sometimes odd.

In [ ]:
for t in [0.2, 0.7, 1.3]:
    print(f"\n===== temperature = {t} =====")
    _ = ask("Invent a name for a gluten-free cookie brand. Reply with the name only.", temperature=t, max_new_tokens=20)

## Experiment C — `do_sample=False` (greedy) vs sampling with different seeds
Greedy gives the identical answer every run. Sampling changes with the seed.

In [ ]:
for s in [1, 2]:
    print(f"\n===== greedy, seed {s} =====")
    _ = ask("Suggest a weekend hobby.", do_sample=False, max_new_tokens=40, seed=s)
for s in [1, 2]:
    print(f"\n===== sampling, seed {s} =====")
    _ = ask("Suggest a weekend hobby.", do_sample=True, max_new_tokens=40, seed=s)

## Checkpoint ✅
You are done with Day 1 when:
1. Step 2 printed a device.
2. Step 3 printed about **494 million** parameters.
3. `ask()` returns sensible text.
4. Experiments A–C behaved as described.

Tomorrow (Day 2) we open up `generate()` and look at tokens and probabilities in more detail.